# Preparación de registros OAI para landing

Permite ver las doce salidas de `oai_load_records` sin escribir en PostgreSQL. La función debe mantenerse alineada con `src/kedro_cic/pipelines/oai_load/nodes.py`. Ejecutar desde `kedro jupyter lab`.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [ ]:
df_records_raw = catalog.load("raw/oai/records#parquet")
env = "dev"  # Cambiar a "full" para inspeccionar todo el batch.
df_records_raw.head()

In [ ]:
def _pick_load_datetime(df: pd.DataFrame) -> pd.Timestamp:
    """Return the batch load timestamp, preserving an existing value if present."""
    for column in ("load_datetime", "_load_datetime"):
        if column not in df.columns:
            continue
        values = pd.to_datetime(df[column], errors="coerce", utc=True).dropna()
        if not values.empty:
            return values.max()
    return pd.Timestamp.now(tz="UTC")

In [ ]:
def _normalize_extract_datetime(df: pd.DataFrame) -> pd.DataFrame:
    """Expose the raw extraction timestamp under the landing contract name."""
    if "extract_datetime" in df.columns:
        return df
    if "_extract_datetime" not in df.columns:
        raise ValueError("OAI records require _extract_datetime")
    return df.rename(columns={"_extract_datetime": "extract_datetime"})

In [ ]:
def oai_load_records(df_records_raw: pd.DataFrame, env: str = "dev") -> pd.DataFrame:
    df_records_raw = _normalize_extract_datetime(df_records_raw.copy())
    load_dt = _pick_load_datetime(df_records_raw)
    if env == "dev":
        df_records_raw = df_records_raw.head(1000)
    def _select(columns):
        return df_records_raw.loc[:, columns].copy()
    def _explode(column):
        base_cols = ["record_id", column, "extract_datetime"]
        missing_cols = [col for col in base_cols if col not in df_records_raw.columns]
        if missing_cols:
            raise ValueError(f"OAI records require columns for {column}: {missing_cols}")
        return (_select(base_cols).explode(column, ignore_index=True).dropna(subset=[column]).assign(load_datetime=load_dt))
    record_columns = ["record_id", "col_id", "title", "date_issued", "extract_datetime", "_context", "_source_key", "_repository_identifier", "_institution_ror", "_base_url", "_metadata_prefix"]
    df_records = _select(record_columns).assign(load_datetime=load_dt)
    df_record_creators = _explode("creators")
    df_record_descriptions = _explode("description")
    df_record_types = _explode("types")
    df_record_identifiers = _explode("identifiers")
    df_record_languages = _explode("languages")
    df_record_subjects = _explode("subjects")
    df_record_publishers = _explode("publishers")
    df_record_relations = _explode("relations")
    df_record_rights = _explode("rights")
    df_record_formats = _explode("formats")
    df_record_sets = _explode("set_id")
    return (df_records, df_record_creators, df_record_descriptions, df_record_types, df_record_identifiers, df_record_languages, df_record_subjects, df_record_publishers, df_record_relations, df_record_rights, df_record_formats, df_record_sets)

In [ ]:
output_names = ["records", "record_creators", "record_descriptions", "record_types", "record_identifiers", "record_languages", "record_subjects", "record_publishers", "record_relations", "record_rights", "record_formats", "record_sets"]
outputs = oai_load_records(df_records_raw, env=env)
prepared = dict(zip(output_names, outputs, strict=True))
assert prepared["records"]["record_id"].notna().all()
assert not prepared["records"]["record_id"].duplicated().any()
assert prepared["records"]["_source_key"].notna().all()
assert all(df["record_id"].notna().all() for df in prepared.values())
pd.DataFrame({"rows": {name: len(df) for name, df in prepared.items()}}).rename_axis("landing_table")

In [ ]:
table_to_preview = "records"  # Cambiar la clave para inspeccionar otra salida.
prepared[table_to_preview].head(20)